# USA Names analysis with BigQuery

This notebook explores the public USA Names dataset with BigQuery, pandas, and matplotlib.

Set GOOGLE_CLOUD_PROJECT or BIGQUERY_PROJECT before running it if the active project cannot be detected.

In [ ]:
# Run this cell if the runtime does not already contain these packages.
%pip install -q google-cloud-bigquery db-dtypes pandas matplotlib

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT') or os.environ.get('BIGQUERY_PROJECT')
if not PROJECT_ID:
    raise ValueError('Set GOOGLE_CLOUD_PROJECT or BIGQUERY_PROJECT.')

client = bigquery.Client(project=PROJECT_ID)
BQ = chr(96)
TABLE = 'bigquery-public-data.usa_names.usa_1910_current'
print(f'Using billing project: {PROJECT_ID}')

## Inspect the public table

In [ ]:
table = client.get_table(TABLE)
print(f'Table: {table.full_table_id}')
print(f'Rows reported by BigQuery: {table.num_rows:,}')
for field in table.schema:
    print(f'- {field.name}: {field.field_type}')

query = f'''
SELECT name, gender, state, year, number
FROM {BQ}{TABLE}{BQ}
ORDER BY year DESC, number DESC
LIMIT 20
'''
preview_df = client.query(query).result().to_dataframe()
preview_df

## Births by year

In [ ]:
query = f'''
SELECT year, SUM(number) AS total_births, COUNT(DISTINCT name) AS unique_names
FROM {BQ}{TABLE}{BQ}
GROUP BY year
ORDER BY year
'''
year_df = client.query(query).result().to_dataframe()
year_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(year_df['year'], year_df['total_births'], color='steelblue')
ax.set_title('Registered births by year')
ax.set_xlabel('Year')
ax.set_ylabel('Births')
ax.grid(alpha=0.25)
plt.show()

## Most popular names

In [ ]:
query = f'''
SELECT name, gender, SUM(number) AS total_births
FROM {BQ}{TABLE}{BQ}
GROUP BY name, gender
ORDER BY total_births DESC
LIMIT 20
'''
top_names_df = client.query(query).result().to_dataframe()
top_names_df

In [ ]:
plot_df = top_names_df.sort_values('total_births')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_df['name'] + ' (' + plot_df['gender'] + ')', plot_df['total_births'], color='darkorange')
ax.set_title('Twenty most common name/gender combinations')
ax.set_xlabel('Births')
plt.tight_layout()
plt.show()

## Selected name trends

In [ ]:
selected_names = ['Michael', 'Jennifer', 'James', 'Emma']
query = f'''
SELECT year, name, SUM(number) AS total_births
FROM {BQ}{TABLE}{BQ}
WHERE name IN UNNEST(@selected_names)
GROUP BY year, name
ORDER BY year, name
'''
job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter('selected_names', 'STRING', selected_names)])
trend_df = client.query(query, job_config=job_config).result().to_dataframe()
trend_wide = trend_df.pivot(index='year', columns='name', values='total_births').fillna(0)
trend_wide.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
trend_wide.plot(ax=ax)
ax.set_title('Selected name trends')
ax.set_xlabel('Year')
ax.set_ylabel('Births')
ax.grid(alpha=0.25)
plt.show()

## State comparison

In [ ]:
query = f'''
SELECT state, SUM(number) AS total_births, COUNT(DISTINCT name) AS unique_names
FROM {BQ}{TABLE}{BQ}
WHERE year >= 2000
GROUP BY state
ORDER BY total_births DESC
'''
state_df = client.query(query).result().to_dataframe()
state_df.head(10)

In [ ]:
plot_df = state_df.head(15).sort_values('total_births')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df['state'], plot_df['total_births'], color='seagreen')
ax.set_title('Births by state since 2000')
ax.set_xlabel('Births')
plt.tight_layout()
plt.show()

## Optional: write a result to BigQuery

In [ ]:
WRITE_RESULTS = False
DESTINATION = f'{PROJECT_ID}.usa_names.births_by_year_notebook'
if WRITE_RESULTS:
    load_job = client.load_table_from_dataframe(year_df, DESTINATION, job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE'))
    load_job.result()
    print(f'Wrote {len(year_df):,} rows to {DESTINATION}')
else:
    print('Result writing is disabled. Set WRITE_RESULTS = True to write the table.')

## Next steps

- Add state and gender filters.
- Recreate a view used by the USA Names Looker Studio dashboard.
- Use Gemini in BigQuery to generate one of the queries and compare it with the SQL here.